# Week 10 — Resonance-gap negative control

`nb10_gap_experiment.ipynb` established that 80/80 gap candidates lie on genuine
tori. What it could **not** establish is the probe ever declining: the gross negative
is a strawman, and the bundle control finds tori because tori are genuinely nearby.

This notebook builds the control that can decline — candidates where a torus should
not exist because a low-order resonance has destroyed it.

**Why famC cannot supply them.** rho/2pi drifts only 2.0e-5 across all 17 members
while the nearest low-order rational, 2/7, sits 7.51e-3 below. Reaching it by
amplitude needs A ~ 10,011 km, 19.4x beyond the family.

**The energy route.** With drho_halo/dC = -3.981297, a step of dC = +1.888e-3 puts
rho_halo itself onto 2/7. Then the resonance sits at *small* torus amplitude instead
of 10,000 km, and kappa < 0 walks a constant-C family straight into it.

**Runtime** ~10 min. Uses `pin='C'` because that is the family in which rho drifts —
`pin='rho'` holds the frequency ratio fixed by construction and can never reach a
resonance.

In [ ]:
import sys, time, contextlib, io
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

import qpit, orbits
from cr3bp_3d import MU_EARTH_MOON as MU, halo_stability
from torus_probe import calibrate_probe, probe_gap, probe_curve, label_from_probe
from gap_sample import resonance_distance

LU = 384400.0
TARGET_P, TARGET_Q = 2, 7        # see section 5 before changing
R27 = TARGET_P / TARGET_Q
t0 = time.time()

d = np.load('../data/qpit_family_L2h29.npz')
KAPPA, DRHO_DC = float(d['kappa']), float(d['drho_halo_dC'])
s0 = np.asarray(d['halo_state0'], float)
T0 = float(d['halo_T'])
print(f"halo #29  C={qpit.jacobi_constant(s0, MU):.9f}  kappa={KAPPA:.6f}  "
      f"drho_halo/dC={DRHO_DC:.6f}")

## 1. Locate the resonant halo

**Pseudo-arclength is mandatory here.** z0 is nearly stationary along the L2 halo
family near #29 — it moves 9e-6 over dC = 1.4e-3 — so the fixed-z0 parameterisation
`halo_corrector` uses is close to singular and natural-parameter continuation in z0
would step past the target or stall. `compute_halo_family` continues in arclength
through [x0, z0, vy0, T_half] and does not care about the fold.

Two stages: a coarse sweep to bracket, then a fine one to land on a chosen margin
`DELTA` above 2/7. DELTA sets where the resonance falls in amplitude, via
A_res = sqrt(DELTA/|kappa|), and 3e-5 puts it near 600 km — inside the range famR
already reached (890 km), so the family can get there.

In [ ]:
def sweep(X, ds, n):
    # Halo family + its centre rotation number. Orbits with no clean centre
    # pair are dropped: they are past the bifurcation and host no torus to seed.
    with contextlib.redirect_stdout(io.StringIO()):
        fam = orbits.compute_halo_family(np.asarray(X, float), ds, n, MU, verbose=False)
    out = []
    for f in fam:
        st = halo_stability(np.asarray(f['state0'], float), f['period'], MU)
        if st['theta'] is not None:
            out.append((f, st['theta'] / (2 * np.pi)))
    return out

DELTA = 3.0e-5                                   # target margin above 2/7
X0 = np.array([s0[0], s0[2], s0[4], T0 / 2])

coarse = sweep(X0, 5e-4, 26)
Cc = np.array([f['jacobi'] for f, _ in coarse])
Rc = np.array([r for _, r in coarse])
slope = np.polyfit(Cc, Rc, 1)[0]
print(f"coarse: {len(coarse)} halos, C {Cc[0]:.9f} -> {Cc[-1]:.9f}")
print(f"  measured drho_halo/dC = {slope:.6f}   (Week 9: {DRHO_DC:.6f}, "
      f"{100*abs(slope-DRHO_DC)/abs(DRHO_DC):.1f}% apart -- the derivative is not constant)")
print(f"  predicted 2/7 crossing at C = {Cc[0] + (R27-Rc[0])/DRHO_DC:.9f}")

j = max(int(np.argmin(np.abs(Rc - (R27 + DELTA)))) - 1, 0)
fine = sweep(coarse[j][0]['X'], 1.0e-5, 60)
k = int(np.argmin([abs(r - (R27 + DELTA)) for _, r in fine]))
halo, rho_h = fine[k]
delta = rho_h - R27
A_res = np.sqrt(delta / abs(KAPPA))

print(f"\nRESONANT HALO  C={halo['jacobi']:.9f}  rho_h/2pi={rho_h:.9f}  "
      f"delta={delta:+.3e}")
print(f"  predicted resonance amplitude A_res = {A_res*LU:.1f} km   [{time.time()-t0:.0f}s]")

## 2. Predictions, before running the continuation

Stated now so the run can falsify them.

1. **The family terminates near A = 604 km.** From rho(A)/2pi = rho_0 + kappa*A^2,
   the resonance is at A_res^2 = delta/|kappa|. Note this must be evaluated
   **self-consistently from any member**: A_res^2 = d_i/|kappa| + A_i^2. Reading
   sqrt(d_i/|kappa|) alone treats each member as if it sat at zero amplitude and
   makes the prediction appear to fall as the family advances — it dropped from 603
   to 526 km in a first pass before this was corrected.

2. **That self-consistent estimate is constant along the family**, to a few tenths of
   a percent, for as long as the kappa*A^2 model holds. Drift in it measures the A^4
   term.

3. **Termination happens at A somewhat BELOW 604 km.** The resonance is a zone, not a
   point: the 1:7 island chain has finite width that grows with amplitude, so the
   family should die on the edge of the zone rather than at its centre. The shortfall
   is a measurement of the island width.

4. **The probe declines beyond termination**, while the identical extrapolation on
   non-resonant famC still converges.

In [ ]:
orbit = dict(state0=np.asarray(halo['state0'], float), period=float(halo['period']))
U_s, T_s, r_s = qpit.seed_from_halo(orbit, MU, 31, 1e-4)
with contextlib.redirect_stdout(io.StringIO()):
    r0 = qpit.converge_torus(U_s, T_s, r_s, MU, tol=1e-12, itmax=20)
print(f"seed torus: {r0['code']}  A={r0['amplitude']*LU:.2f} km  "
      f"rho/2pi={r0['rho']/(2*np.pi):.9f}  d(2/7)={r0['rho']/(2*np.pi)-R27:+.3e}")

with contextlib.redirect_stdout(io.StringIO()):
    famRes = qpit.continue_torus_family(r0['U0'], r0['T'], r0['rho'], MU,
                                        pin='C', n_steps=26, offgrid_every=0,
                                        verbose=False)
mem = famRes['members']
print(f"\ncontinuation: {len(mem)} members, terminated = {famRes['terminated']}"
      f"   [{time.time()-t0:.0f}s]")

In [ ]:
A_m = np.array([m['amplitude'] for m in mem])
R_m = np.array([m['rho'] for m in mem]) / (2 * np.pi)
d_m = R_m - R27
A_res_sc = np.sqrt(d_m / abs(KAPPA) + A_m**2)          # self-consistent (pred. 1 & 2)

print(f"{'i':>3} {'A [km]':>9} {'rho/2pi':>13} {'d(2/7)':>11} {'A_res [km]':>11}")
for i in range(len(mem)):
    print(f"{i:3d} {A_m[i]*LU:9.2f} {R_m[i]:13.9f} {d_m[i]:+11.3e} {A_res_sc[i]*LU:11.1f}")
print(f"\nA_res self-consistent: {A_res_sc.mean()*LU:.1f} +/- {A_res_sc.std()*LU:.1f} km"
      f"  (spread {100*A_res_sc.std()/A_res_sc.mean():.2f}%)")
crossed = np.any(d_m < 0)
print(f"family reached A = {A_m[-1]*LU:.1f} km, terminated by '{famRes['terminated']}'")
print(f"crossed rho = {TARGET_P}/{TARGET_Q} without terminating: {crossed}"
      f"   <- prediction 3 fails if True")

fig, ax = plt.subplots(figsize=(6.6, 4.6))
ax.plot(A_m * LU, d_m, 'o-', ms=4, label='family (pin=C)')
Af = np.linspace(0, A_res_sc.mean() * 1.05, 300)
ax.plot(Af * LU, d_m[0] + KAPPA * (Af**2 - A_m[0]**2), 'k--', lw=1,
        label=r'$\kappa A^2$ model')
ax.axhline(0, color='tab:red', lw=1.2, label='2/7 resonance')
ax.axvline(A_res_sc.mean() * LU, color='tab:red', ls=':', lw=1,
           label=f'predicted {A_res_sc.mean()*LU:.0f} km')
ax.set_xlabel('amplitude [km]'); ax.set_ylabel(r'$\rho/2\pi - 2/7$')
ax.set_title('A constant-energy family walking into a 1:7 resonance')
ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. Negative candidates, with the extrapolation confound controlled

Candidates come from extrapolating the family tangent past the last member —
`probe_gap` with alpha > 1 on the final pair.

**The confound.** Linear extrapolation error grows as alpha^2, so a failure at large
alpha could mean "no torus here" or simply "the candidate curve is a bad guess".
Those are different claims and the experiment cannot tell them apart on its own.

**The control.** Run the identical extrapolation, at identical alpha, on the final
pair of non-resonant famC. Same construction, same interpolation error, no resonance
nearby. If the resonant family fails where famC converges, the difference is the
resonance.

In [ ]:
ALPHAS_EXT = [1.0, 1.25, 1.5, 2.0, 3.0, 4.0]
cal = calibrate_probe([mem[0], mem[len(mem)//2], mem[-1]], MU,
                      perturb=1e-6, offgrid=False)
print(f"threshold dist_max <= {cal['dist_max']:.3e}   [{time.time()-t0:.0f}s]")

def extrapolate(pair, tag):
    rows = probe_gap(pair[0], pair[1], MU, ALPHAS_EXT, offgrid=False)
    for p in rows:
        p.update(arm=tag, d_res=resonance_distance(p['rho_final'], q_max=7)[0])
    return rows

def as_rec(m):
    return dict(U0=m['U0'], T=m['T'], rho=m['rho'], C=m['C'],
                amplitude=m['amplitude'])

res_ext = extrapolate((as_rec(mem[-2]), as_rec(mem[-1])), 'resonant')

famC_arch = np.load('../data/qpit_family_L2h29.npz')
o = np.argsort(famC_arch['C_A'])
ctrl_pair = tuple(dict(U0=famC_arch['C_U'][i], T=float(famC_arch['C_T'][i]),
                       rho=float(famC_arch['C_rho'][i]), C=float(famC_arch['C_C'][i]),
                       amplitude=float(famC_arch['C_A'][i])) for i in o[-2:])
ctl_ext = extrapolate(ctrl_pair, 'nonresonant')

print(f"\n{'alpha':>6} | {'RESONANT':^34} | {'NON-RESONANT (control)':^34}")
print(f"{'':>6} | {'conv':>5} {'dist_max':>11} {'d(2/7)':>10} | {'conv':>5} {'dist_max':>11} {'d(2/7)':>10}")
for pr, pc in zip(res_ext, ctl_ext):
    print(f"{pr['alpha']:6.2f} | {str(pr['converged']):>5} {pr['dist_max']:11.3e} "
          f"{pr['d_res']:10.2e} | {str(pc['converged']):>5} {pc['dist_max']:11.3e} "
          f"{pc['d_res']:10.2e}")
print(f"[{time.time()-t0:.0f}s]")

## 4. Score against resonance proximity

**Generation, not analysis.** This plot exists to confirm the negatives are real and
that the score varies with resonance distance — i.e. that the dataset contains the
signal Week 12 will need. Connecting that signal to *classifier* accuracy is Week 12
and is not done here.

In [ ]:
allp = res_ext + ctl_ext
fig, ax = plt.subplots(figsize=(6.6, 4.4))
for tag, col, mk in (('resonant', 'tab:red', 'o'), ('nonresonant', 'tab:blue', 's')):
    sel = [p for p in allp if p['arm'] == tag]
    ax.semilogy([p['alpha'] for p in sel], [max(p['dist_max'], 1e-9) for p in sel],
                mk + '-', color=col, ms=6, label=tag)
    fail = [p for p in sel if not p['converged']]
    if fail:
        ax.semilogy([p['alpha'] for p in fail],
                    [max(p['dist_max'], 1e-9) for p in fail],
                    'x', color='k', ms=11, mew=2,
                    label='did not converge' if tag == 'resonant' else None)
ax.axhline(cal['dist_max'], color='0.4', ls='--', lw=1, label='threshold')
ax.set_xlabel(r'extrapolation $\alpha$ past the last member')
ax.set_ylabel('probe dist_max')
ax.set_title('Same construction, same interpolation error, different outcome')
ax.grid(alpha=.3, which='both'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Append to the Week 10 dataset

Same two-table schema as the gap notebook. `arm` distinguishes the source, `d_res`
carries resonance proximity, and `censored` marks samples where `dist_max` records
how far Newton travelled rather than how far the nearest torus is.

In [ ]:
CENSOR = 1e-3
family_probes = []
for m in mem:
    p = probe_curve(m['U0'], m['T'], m['rho'], MU, C_target=m['C'], offgrid=False)
    p.update(arm='resonant_family', alpha=np.nan,
             d_res=resonance_distance(p['rho_final'], q_max=7)[0])
    family_probes.append(p)

probes = family_probes + res_ext + ctl_ext
rng = np.random.default_rng(20)
N_THETA = 8
X, cid, th = [], [], []
for i, p in enumerate(probes):
    if not p['converged']:
        continue
    t = rng.uniform(0, 2*np.pi, N_THETA)
    X.append(qpit.evaluate_curve(p['result']['U0'], t))
    cid.append(np.full(N_THETA, i)); th.append(t)
X = np.vstack(X); cid = np.concatenate(cid); th = np.concatenate(th)

f = lambda k: np.array([float(p.get(k, np.nan)) for p in probes])
np.savez('../data/qpit_dataset_w10_resonance.npz',
         states=X, curve_id=cid, theta=th,
         arm=np.array([str(p['arm']) for p in probes]),
         score_C=f('dist_max'), d_res=f('d_res'), alpha=f('alpha'),
         amp_cand=f('amp_cand'), amp_final=f('amp_final'),
         rho_final=f('rho_final'), C_target=f('C_target'), C_seed=f('C_seed'),
         res_inf=f('res_inf'), damp_rel=f('damp_rel'), reproj_rel=f('reproj_rel'),
         converged=np.array([p['converged'] for p in probes]),
         censored=np.array([(not p['converged']) or p['dist_max'] > CENSOR
                            for p in probes]),
         label_B=np.array([bool(label_from_probe(p, cal)) for p in probes]),
         halo_C=halo['jacobi'], halo_rho=rho_h, delta=delta,
         A_res_pred=A_res_sc.mean(), A_terminated=A_m[-1],
         threshold_dist_max=cal['dist_max'], censor_level=CENSOR,
         resonance_pq=np.array([2, 7]), mu=MU, n_theta=N_THETA)

n_neg = sum(1 for p in probes if not p['converged'])
print(f"{len(probes)} curves, {X.shape[0]} states, {n_neg} genuine negatives")
print(f"saved   [{time.time()-t0:.0f}s total]")

## 5. Result — predictions 1 and 2 hold, 3 and 4 fail

**Confirmed.** A_res self-consistent = **604.0 +/- 0.5 km, spread 0.09%**, constant
across all 27 members. The kappa*A^2 model is excellent over a 55x amplitude range,
and the crossing energy predicted from Week 9's drho_halo/dC (C = 3.062222) matched
the measured one (3.062225) to three parts per million.

**Falsified.** The family did not terminate at the resonance. It crossed 2/7 cleanly
between members 19 (d = +2.48e-06) and 20 (d = -2.58e-06) and continued to 968 km,
stopping only because it ran out of requested steps. Every extrapolated candidate
converged, resonant and control alike, so the run produced **no negatives at all**.

**Diagnosis: q = 7 is not low order.** Resonance zone width falls off very steeply
with the denominator — the 1:7 islands are far narrower than the 56 km continuation
step, and plausibly narrower than anything this solver could resolve. The family
stepped over the gap without ever entering it. Choosing 2/7 because it was
*nearest in rho* was the error; what matters is *smallest q*, and those are not the
same ordering.

This is a real result rather than a failed run: it bounds where torus destruction is
detectable, which is exactly what secondary question 2 needs to be posed correctly.

**Retarget to 1/4.** Verified reachable: `compute_halo_family` at ds = 2e-3 crosses
rho_halo/2pi = 0.25 between members 20 and 22, near C = 3.0716, in ~2 s, with a clean
centre pair at every step. kappa < 0 walks a family down into it, so the mechanism is
the same and only the target moves. Set `TARGET_P, TARGET_Q = 1, 4` and widen the
coarse sweep to ds = 2e-3, n = 40.

1/3 is stronger still (q = 3) but sits *above* rho_0, so kappa walks away from it; it
needs a halo above 1/3 at C < 3.0503 and is the fallback if 1/4 also proves too thin.

## 5. Result — predictions 1 and 2 hold, 3 and 4 fail

**Confirmed.** A_res self-consistent = **604.0 +/- 0.5 km, spread 0.09%**, constant
across all 27 members. The kappa*A^2 model is excellent over a 55x amplitude range,
and the crossing energy predicted from Week 9's drho_halo/dC (C = 3.062222) matched
the measured one (3.062225) to three parts per million.

**Falsified.** The family did not terminate at the resonance. It crossed 2/7 cleanly
between members 19 (d = +2.48e-06) and 20 (d = -2.58e-06) and continued to 968 km,
stopping only because it ran out of requested steps. Every extrapolated candidate
converged, resonant and control alike, so the run produced **no negatives at all**.

**Diagnosis: q = 7 is not low order.** Resonance zone width falls off very steeply
with the denominator — the 1:7 islands are far narrower than the 56 km continuation
step, and plausibly narrower than anything this solver could resolve. The family
stepped over the gap without ever entering it. Choosing 2/7 because it was
*nearest in rho* was the error; what matters is *smallest q*, and those are not the
same ordering.

This is a real result rather than a failed run: it bounds where torus destruction is
detectable, which is exactly what secondary question 2 needs to be posed correctly.

**Retarget to 1/4.** Verified reachable: `compute_halo_family` at ds = 2e-3 crosses
rho_halo/2pi = 0.25 between members 20 and 22, near C = 3.0716, in ~2 s, with a clean
centre pair at every step. kappa < 0 walks a family down into it, so the mechanism is
the same and only the target moves. Set `TARGET_P, TARGET_Q = 1, 4` and widen the
coarse sweep to ds = 2e-3, n = 40.

1/3 is stronger still (q = 3) but sits *above* rho_0, so kappa walks away from it; it
needs a halo above 1/3 at C < 3.0503 and is the fallback if 1/4 also proves too thin.

## What this establishes

The machinery is built and validated: the halo family lands on a chosen resonance to
3 parts per million in C, the torus family walks in as kappa predicts, and the
extrapolation confound is controlled by a matched non-resonant arm.

What is **not** yet established is the probe declining. At 2/7 the resonance zone was
too thin to enter, so the dataset still has no physical negatives. That waits on the
1/4 retarget in section 5, which is a two-line change to a notebook that otherwise
runs unaltered.

**Not this week:** any training, the phase-space failure map, connecting `d_res` to
classifier accuracy. Those are Weeks 11-12.

**Open, for the Sunday agenda:** the shortfall between predicted A_res and actual
termination is an island-width measurement obtained almost for free. Whether it is
worth resolving properly — by repeating at several DELTA values — is a Phase 5
question, and Phase 5 is the designated cut.